# DESC ELAsTiCC2 — Demo 3 : Light-curve fitting with band renormalisation

This notebook fits SNIa light curves from ELAsTiCC2 using a **single analytic template** shared
across all six LSST bands (u, g, r, i, z, y).  The r-band flux is taken as the reference; the
flux in every other band is rescaled by a multiplicative colour factor `A_b`.

Two template functions are available (choose via `FIT_MODEL`):

| `FIT_MODEL` | Description |
|---|---|
| `'poly2'` | Degree-2 polynomial in `(t − t_max)` × a width (stretch) parameter |
| `'weibull'` | Generalised asymmetric Weibull (β, γ shape parameters) |

### Two fitting strategies

**Part 1 — Individual fits**  
Each event is fitted independently.  All parameters (`A_b`, `t_max`, `stretch`, shape) are free per event.
Results are displayed as a grid of renormalised + overplotted model curves.

**Part 2 — Global fit**  
A two-step approach:  
* Per-event parameters (`t_max`, `stretch`, shape) are kept event-specific.  
* Colour factors `A_b` are **shared** across all selected events and optimised jointly.

**Configurable parameters** are in the `Parameters` cell below.


## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit, minimize
from scipy.special import gamma as gamma_func

# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")



## 1 · Parameters

In [ ]:
# ── Data selection ─────────────────────────────────────────────────────────────
OBJ_CLASS      = 'SNIa-SALT3'   # SNANA class
Z_MIN          = 0.1            # redshift lower bound
Z_MAX          = 0.5            # redshift upper bound
FILE_NUM       = 1              # PHOT file index (1–40; None = all)
MIN_DETECTIONS = 8              # minimum detected points per object
DETECTED_ONLY  = True           # use only detected points (PHOTFLAG & photflag_detect)
N_CURVES       = 100            # number of events for Part 1 (individual fits)
N_GLOBAL       = 100             # number of events for Part 2 (global fit)
RANDOM_SEED    = 42

# ── Model choice ───────────────────────────────────────────────────────────────
# 'poly2'   : degree-2 polynomial template
# 'weibull' : asymmetric Weibull / generalised beta profile
#FIT_MODEL = 'poly2'             # <-- change here
FIT_MODEL = 'weibull'
# ── Reference band ─────────────────────────────────────────────────────────────
BANDS      = ['u', 'g', 'r', 'i', 'z', 'Y']
REF_BAND   = 'r'               # r-band flux = reference (A_r ≡ 1)

# ── Data path ──────────────────────────────────────────────────────────────────
DATA_DIR   = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX = "ELASTICC2_TRAIN_02_"

# ── Plot colours per band ──────────────────────────────────────────────────────
BAND_COLORS = {
    'u': '#cc0ccc',
    'g': '#00cc44',
    'r': '#cc0000',
    'i': '#ff4400',
    'z': '#886600',
    'Y': '#442200'
}

# ── Display ────────────────────────────────────────────────────────────────────
NCOLS      = 4
NROWS_P1   = math.ceil(N_CURVES / NCOLS)

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"Model : {FIT_MODEL}   |   ref band : {REF_BAND}   |   N_CURVES={N_CURVES}   |   N_GLOBAL={N_GLOBAL}")

## 2 · Template functions

### 2.1 Degree-2 polynomial

$$
f_{\rm poly2}(t) = F_{\rm peak}\,\max\!\left(0,\; 1 - \left(\frac{t - t_{\rm max}}{s}\right)^{\!2}\right)
$$

where $s$ is the stretch (half-width at zero flux) and $F_{\rm peak}$ the peak flux.

### 2.2 Asymmetric Weibull / generalised profile

$$
f_{\rm W}(t) = F_{\rm peak}\,\exp\!\left[-\left|\frac{t - t_{\rm max}}{s\,\sigma(t)}\right|^{\!\beta}\right]
$$

with an asymmetric width $\sigma(t) = 1$ for $t < t_{\rm max}$ (rise) and $\sigma(t) = \gamma$ for $t \ge t_{\rm max}$ (decay).  
$\beta$ controls the peakiness, $\gamma$ the rise/fall asymmetry.

In [ ]:
def template_poly2(t: np.ndarray, t_max: float, s: float) -> np.ndarray:
    """Parabolic template, clipped to non-negative values.

    Parameters
    ----------
    t     : time array (MJD)
    t_max : peak time
    s     : stretch (half-width at zero crossing), must be > 0

    Returns
    -------
    Normalised profile in [0, 1].
    """
    return np.maximum(0.0, 1.0 - ((t - t_max) / s) ** 2)


def template_weibull(t: np.ndarray, t_max: float, s: float,
                     beta: float = 2.0, gamma_asym: float = 1.5) -> np.ndarray:
    """Asymmetric Weibull-like profile normalised to peak = 1.

    Parameters
    ----------
    t          : time array (MJD)
    t_max      : peak time
    s          : characteristic width (stretch), must be > 0
    beta       : shape exponent (peakiness), must be > 0
    gamma_asym : asymmetry ratio (decay/rise width), must be > 0

    Returns
    -------
    Normalised profile in (0, 1].
    """
    dt = t - t_max
    sigma = np.where(dt < 0, 1.0, float(gamma_asym))  # rise vs decay
    return np.exp(-np.abs(dt / (s * sigma)) ** float(beta))


# ── Quick sanity plot ──────────────────────────────────────────────────────────
t_demo = np.linspace(-30, 60, 300)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), tight_layout=True)
axes[0].plot(t_demo, template_poly2(t_demo, 0, 20), 'b-', lw=2, label='poly2  s=20')
axes[0].plot(t_demo, template_poly2(t_demo, 0, 15), 'b--', lw=2, label='poly2  s=15')
axes[0].set_title('Parabolic template'); axes[0].set_xlabel('t − t_max'); axes[0].legend()
axes[1].plot(t_demo, template_weibull(t_demo, 0, 15, beta=2.0, gamma_asym=1.5), 'r-',  lw=2, label='β=2, γ=1.5')
axes[1].plot(t_demo, template_weibull(t_demo, 0, 15, beta=1.5, gamma_asym=2.0), 'r--', lw=2, label='β=1.5, γ=2')
axes[1].set_title('Weibull template'); axes[1].set_xlabel('t − t_max'); axes[1].legend()
for ax in axes: ax.axhline(0, color='k', lw=0.5); ax.set_ylabel('Normalised flux')
plt.suptitle('Template shapes', fontsize=12)
plt.show()

## 3 · Model building: multi-band flux

For a given event, the predicted flux in band $b$ at time $t$ is

$$
F_b(t) = A_b \cdot F_{\rm peak} \cdot \phi(t;\, t_{\rm max},\, s,\, \ldots)
$$

with $A_r \equiv 1$ (reference band).  
The free parameters are:  
* $F_{\rm peak}$ : peak flux in the r-band  
* $A_b\,(b \neq r)$ : 5 colour factors (u, g, i, z, Y)  
* $t_{\rm max}$ : peak time  
* $s$ : stretch (width)  
* *(Weibull only)* $\beta$, $\gamma_{\rm asym}$ : shape parameters

In [ ]:
# Fixed ordering of colour-factor bands (all except REF_BAND)
COLOR_BANDS = [b for b in BANDS if b != REF_BAND]  # ['u','g','i','z','Y']


def build_model_flux(band_arr: np.ndarray, mjd_arr: np.ndarray,
                     A_dict: dict, F_peak: float,
                     t_max: float, s: float,
                     beta: float = 2.0, gamma_asym: float = 1.5,
                     model: str = 'poly2') -> np.ndarray:
    """Compute model flux for a set of (band, mjd) measurements.

    Parameters
    ----------
    band_arr   : array of band labels (str)
    mjd_arr    : array of MJD values
    A_dict     : dict {band: amplitude factor}; REF_BAND should map to 1.0
    F_peak     : r-band peak flux
    t_max      : peak time
    s          : stretch width
    beta, gamma_asym : Weibull shape parameters (ignored for poly2)
    model      : 'poly2' or 'weibull'

    Returns
    -------
    Predicted flux array.
    """
    if model == 'poly2':
        phi = template_poly2(mjd_arr, t_max, s)
    elif model == 'weibull':
        phi = template_weibull(mjd_arr, t_max, s, beta=beta, gamma_asym=gamma_asym)
    else:
        raise ValueError(f"Unknown model '{model}'. Choose 'poly2' or 'weibull'.")

    flux_pred = np.zeros(len(mjd_arr))
    for b, A in A_dict.items():
        mask = (band_arr == b)
        flux_pred[mask] = A * F_peak * phi[mask]
    return flux_pred


print("Model builder ready.")
print(f"  Reference band : {REF_BAND}")
print(f"  Colour bands   : {COLOR_BANDS}")
print(f"  Template       : {FIT_MODEL}")

## 4 · Individual-fit helper

The chi-squared objective is minimised with `scipy.optimize.minimize` (L-BFGS-B),
which allows box constraints on all parameters.

In [ ]:
def fit_single_event(ltcv_df: pd.DataFrame, model: str = 'poly2',
                     ref_band: str = 'r', color_bands: list = None,
                     bands: list = None) -> dict:
    """Fit a single-event light curve across all bands.

    Returns a dict with keys:
        'success', 'params', 'chi2', 'ndof',
        'F_peak', 't_max', 's', 'A_dict',
        ['beta', 'gamma_asym'] if model == 'weibull'
    """
    if color_bands is None:
        color_bands = COLOR_BANDS
    if bands is None:
        bands = BANDS

    # ── Prepare data arrays ────────────────────────────────────────────────────
    mask_det = ltcv_df['FLUXCALERR'] > 0
    df = ltcv_df[mask_det].copy()
    mjd   = df['MJD'].values
    flux  = df['FLUXCAL'].values
    ferr  = df['FLUXCALERR'].values
    bands_arr = df['BAND'].values

    if len(mjd) < 5:
        return {'success': False, 'reason': 'not enough points'}

    # ── Initial guess ─────────────────────────────────────────────────────────
    # Use the r-band points to estimate t_max and peak flux
    r_mask = bands_arr == ref_band
    if r_mask.sum() < 2:
        # Fall back to all bands for initial guess
        r_mask = np.ones(len(mjd), dtype=bool)
    idx_peak  = np.argmax(flux[r_mask])
    t_max0    = mjd[r_mask][idx_peak]
    F_peak0   = max(flux[r_mask][idx_peak], 1.0)
    s0        = 20.0   # days, typical SNIa width

    # Colour factors: ratio of median flux in each band vs r-band
    A0 = {}
    r_flux_near_peak = flux[(r_mask) & (np.abs(mjd - t_max0) < s0)]
    ref_med = np.median(r_flux_near_peak) if len(r_flux_near_peak) > 0 else F_peak0
    ref_med = max(ref_med, 1.0)
    A0[ref_band] = 1.0
    for b in color_bands:
        b_mask = bands_arr == b
        if b_mask.sum() > 0:
            b_flux_near = flux[(b_mask) & (np.abs(mjd - t_max0) < s0)]
            A0[b] = max(np.median(b_flux_near) / ref_med, 0.05) if len(b_flux_near) > 0 else 0.5
        else:
            A0[b] = 0.5

    # ── Parameter vector layout ────────────────────────────────────────────────
    # p = [F_peak, t_max, s, A_u, A_g, A_i, A_z, A_Y,  (beta, gamma_asym for weibull)]
    p0 = [F_peak0, t_max0, s0] + [A0[b] for b in color_bands]
    bounds_low  = [0.0,  mjd.min()-5,  2.0] + [0.01]*len(color_bands)
    bounds_high = [np.inf, mjd.max()+5, 200.0] + [20.0]*len(color_bands)

    if model == 'weibull':
        p0          += [2.0, 1.5]
        bounds_low  += [0.5, 0.2]
        bounds_high += [6.0, 8.0]

    # ── Objective (chi-squared) ────────────────────────────────────────────────
    def chi2(p):
        F_pk = p[0]; t_mx = p[1]; s_ = abs(p[2])
        A_vals = p[3:3+len(color_bands)]
        A_d = {ref_band: 1.0}
        for i, b in enumerate(color_bands):
            A_d[b] = A_vals[i]
        kw = {}
        if model == 'weibull':
            kw = {'beta': p[-2], 'gamma_asym': p[-1]}
        flux_pred = build_model_flux(bands_arr, mjd, A_d, F_pk, t_mx, s_, model=model, **kw)
        residuals = (flux - flux_pred) / ferr
        return float(np.sum(residuals**2))

    # ── Minimise ───────────────────────────────────────────────────────────────
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = minimize(
            chi2, p0,
            method='L-BFGS-B',
            bounds=list(zip(bounds_low, bounds_high)),
            options={'maxiter': 2000, 'ftol': 1e-12, 'gtol': 1e-8}
        )

    p_opt = res.x
    F_pk_opt = p_opt[0]; t_mx_opt = p_opt[1]; s_opt = abs(p_opt[2])
    A_d_opt = {ref_band: 1.0}
    for i, b in enumerate(color_bands):
        A_d_opt[b] = p_opt[3+i]

    ndof = len(mjd) - len(p0)
    result = {
        'success':  res.success,
        'chi2':     res.fun,
        'chi2_red': res.fun / max(ndof, 1),
        'ndof':     ndof,
        'F_peak':   F_pk_opt,
        't_max':    t_mx_opt,
        's':        s_opt,
        'A_dict':   A_d_opt,
        'params':   p_opt,
    }
    if model == 'weibull':
        result['beta']      = p_opt[-2]
        result['gamma_asym']= p_opt[-1]
    return result


print("Single-event fitter ready.")

## 5 · Load data

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading HEAD for {OBJ_CLASS}...")
head  = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading truth for {OBJ_CLASS}...")
truth = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")

In [ ]:
# ── Filter: redshift + minimum detections ─────────────────────────────────────
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)

truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
subset = truth_counts[
    (truth_counts['ZCMB'] >= Z_MIN) &
    (truth_counts['ZCMB'] <  Z_MAX) &
    (truth_counts['ndetect'] >= MIN_DETECTIONS)
].copy()

print(f"{len(subset)} objects pass selection (z∈[{Z_MIN},{Z_MAX}), ndet≥{MIN_DETECTIONS}).")

---
# Part 1 · Individual light-curve fits

In [ ]:
# ── Select N_CURVES events ────────────────────────────────────────────────────
n_avail = min(N_CURVES, len(subset))
chosen_idx   = rng.choice(len(subset), size=n_avail, replace=False)
chosen_snids = subset['SNID'].values[chosen_idx]
print(f"Selected {n_avail} SNIDs for individual fitting.")

In [ ]:
# ── Run individual fits ───────────────────────────────────────────────────────
fit_results_p1 = {}
for snid in chosen_snids:
    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]
    if DETECTED_ONLY:
        ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]
    result = fit_single_event(ltcv, model=FIT_MODEL, ref_band=REF_BAND,
                              color_bands=COLOR_BANDS, bands=BANDS)
    fit_results_p1[snid] = result
    status = '✓' if result.get('success') else '✗'
    chi2r  = result.get('chi2_red', float('nan'))
    print(f"  SNID {snid:8d}  {status}  χ²/dof = {chi2r:.2f}  "
          f"t_max={result.get('t_max', np.nan):.1f}  "
          f"s={result.get('s', np.nan):.1f} d")

In [ ]:
# ── Plot: renormalised light curves + individual fits ─────────────────────────
FIG_W, FIG_H = 5.5, 4.2
nrows = math.ceil(n_avail / NCOLS)

fig, axes = plt.subplots(nrows, NCOLS,
                         figsize=(FIG_W * NCOLS, FIG_H * nrows),
                         tight_layout=True)
axes_flat = np.array(axes).flatten()

t_model_dense = np.linspace(-60, 100, 500)   # relative time axis for model curve

for idx, snid in enumerate(chosen_snids):
    ax  = axes_flat[idx]
    res = fit_results_p1[snid]

    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]
    if DETECTED_ONLY:
        ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

    z_val = truth[truth['SNID'] == snid]['ZCMB'].values
    z_val = z_val[0] if len(z_val) else float('nan')

    if not res.get('success', False) and 'F_peak' not in res:
        ax.set_title(f"SNID {snid}\nFit failed", fontsize=9, color='red')
        continue

    t_max   = res['t_max']
    s       = res['s']
    F_peak  = res['F_peak']
    A_dict  = res['A_dict']
    chi2r   = res.get('chi2_red', float('nan'))

    # ── Plot data, renormalised by A_b * F_peak so all bands map to a common scale
    for band in BANDS:
        bdf = ltcv[ltcv['BAND'] == band]
        if len(bdf) == 0:
            continue
        A_b = A_dict.get(band, 1.0)
        norm_factor = max(A_b * F_peak, 1e-6)   # renormalise to r-band scale
        dt   = bdf['MJD'].values - t_max
        f_n  = bdf['FLUXCAL'].values / norm_factor
        fe_n = bdf['FLUXCALERR'].values / norm_factor
        ax.errorbar(dt, f_n, yerr=fe_n,
                    color=BAND_COLORS.get(band, 'gray'),
                    linestyle='None', marker='o', markersize=3,
                    capsize=2, label=band)

    # ── Overplot the template curve (same for all bands after renorm)
    if FIT_MODEL == 'poly2':
        phi_model = template_poly2(t_max + t_model_dense, t_max, s)
    else:
        phi_model = template_weibull(t_max + t_model_dense, t_max, s,
                                     beta=res.get('beta', 2.0),
                                     gamma_asym=res.get('gamma_asym', 1.5))
    ax.plot(t_model_dense, phi_model, 'k-', lw=1.5, label='model', zorder=5)
    ax.axhline(0, color='k', lw=0.4, ls='--')

    ax.set_xlim(t_model_dense[0], t_model_dense[-1])
    ax.set_title(
        f"SNID {snid}  z={z_val:.3f}\n"
        f"t_max={t_max:.1f}  s={s:.1f} d  χ²/dof={chi2r:.2f}",
        fontsize=8
    )
    ax.set_xlabel(r"$t - t_{\rm max}$ [days]", fontsize=8)
    ax.set_ylabel(r"Flux / $(A_b F_{\rm peak})$", fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.legend(fontsize=6, ncol=3, loc='upper right')

for idx in range(n_avail, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    f"Part 1 · Individual fits ({FIT_MODEL}) — {OBJ_CLASS}\n"
    f"z ∈ [{Z_MIN},{Z_MAX})  |  ndet ≥ {MIN_DETECTIONS}  |  ref band = {REF_BAND}",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

### Distribution of individual-fit parameters

In [ ]:
# Collect results into a DataFrame for inspection
rows = []
for snid, res in fit_results_p1.items():
    if not res.get('F_peak'):
        continue
    row = {'SNID': snid,
           'chi2_red': res.get('chi2_red', np.nan),
           'F_peak':   res['F_peak'],
           't_max':    res['t_max'],
           's':        res['s']}
    for b in COLOR_BANDS:
        row[f'A_{b}'] = res['A_dict'].get(b, np.nan)
    if FIT_MODEL == 'weibull':
        row['beta']       = res.get('beta', np.nan)
        row['gamma_asym'] = res.get('gamma_asym', np.nan)
    rows.append(row)

df_p1 = pd.DataFrame(rows)
print(df_p1.to_string(index=False, float_format='{:.3f}'.format))

---
# Part 2 · Global fit — shared colour factors

Strategy:
1. Use the individual fits from Part 1 (or a larger sample) as initial per-event parameters.  
2. Define a **joint chi-squared** over all events that depends on:  
   - Shared parameters: `A_u, A_g, A_i, A_z, A_Y` (5 colour factors)  
   - Per-event parameters: `F_peak_k, t_max_k, s_k` (and `beta_k, gamma_asym_k` for Weibull)  
3. Minimise with a block-coordinate descent:
   * **Step A** — fix colour factors, update each event's parameters independently (fast inner loop).  
   * **Step B** — fix all per-event parameters, update shared colour factors.  
   * Iterate until convergence.

In [ ]:
# ── Select N_GLOBAL events ────────────────────────────────────────────────────
n_global = min(N_GLOBAL, len(subset))
# Re-use the same rng to get a fresh independent draw
rng_global  = np.random.default_rng(seed=RANDOM_SEED + 1)
idx_global  = rng_global.choice(len(subset), size=n_global, replace=False)
snids_global = subset['SNID'].values[idx_global]

# Pre-extract light curves
ltcvs_global = {}
for snid in snids_global:
    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]
    if DETECTED_ONLY:
        ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]
    ltcvs_global[snid] = ltcv

print(f"Global fit sample: {n_global} events.")

In [ ]:
def fit_event_fixed_colors(ltcv_df: pd.DataFrame,
                           A_shared: dict,
                           model: str = 'poly2',
                           ref_band: str = 'r',
                           color_bands: list = None) -> dict:
    """Fit a single event with FIXED colour factors A_shared.

    Optimises only F_peak, t_max, s (and beta, gamma_asym for Weibull).
    """
    if color_bands is None:
        color_bands = COLOR_BANDS

    mask_det = ltcv_df['FLUXCALERR'] > 0
    df = ltcv_df[mask_det].copy()
    mjd  = df['MJD'].values
    flux = df['FLUXCAL'].values
    ferr = df['FLUXCALERR'].values
    bands_arr = df['BAND'].values

    if len(mjd) < 4:
        return {'success': False}

    r_mask  = bands_arr == ref_band
    if r_mask.sum() < 2: r_mask = np.ones(len(mjd), dtype=bool)
    idx_pk  = np.argmax(flux[r_mask])
    t_max0  = mjd[r_mask][idx_pk]
    F_peak0 = max(flux[r_mask][idx_pk], 1.0)

    A_full = {ref_band: 1.0, **{b: A_shared[b] for b in color_bands}}

    p0 = [F_peak0, t_max0, 20.0]
    bl = [0.0, mjd.min()-5, 2.0]
    bh = [np.inf, mjd.max()+5, 200.0]
    if model == 'weibull':
        p0 += [2.0, 1.5]; bl += [0.5, 0.2]; bh += [6.0, 8.0]

    def chi2(p):
        kw = {}
        if model == 'weibull':
            kw = {'beta': p[3], 'gamma_asym': p[4]}
        fp = build_model_flux(bands_arr, mjd, A_full,
                              p[0], p[1], abs(p[2]), model=model, **kw)
        return float(np.sum(((flux - fp) / ferr)**2))

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = minimize(chi2, p0, method='L-BFGS-B',
                       bounds=list(zip(bl, bh)),
                       options={'maxiter': 500, 'ftol': 1e-10})

    result = {'success': res.success, 'chi2': res.fun,
              'F_peak': res.x[0], 't_max': res.x[1], 's': abs(res.x[2])}
    if model == 'weibull':
        result['beta'] = res.x[3]; result['gamma_asym'] = res.x[4]
    return result


def update_shared_colors(snids: list, ltcvs: dict,
                         per_event: dict,
                         A_current: dict,
                         model: str = 'poly2',
                         ref_band: str = 'r',
                         color_bands: list = None) -> dict:
    """Optimise shared colour factors given fixed per-event parameters."""
    if color_bands is None:
        color_bands = COLOR_BANDS

    # Stack all observations
    all_bands, all_mjd, all_flux, all_ferr = [], [], [], []
    all_phi, all_Fpeak = [], []

    for snid in snids:
        ev = per_event.get(snid)
        if ev is None or not ev.get('F_peak'):
            continue
        df = ltcvs[snid]
        mask = df['FLUXCALERR'] > 0
        df = df[mask]
        mjd  = df['MJD'].values
        flux = df['FLUXCAL'].values
        ferr = df['FLUXCALERR'].values
        bands_arr = df['BAND'].values

        t_max = ev['t_max']; s = ev['s']; F_pk = ev['F_peak']
        kw = {}
        if model == 'weibull':
            kw = {'beta': ev.get('beta', 2.0), 'gamma_asym': ev.get('gamma_asym', 1.5)}
        if model == 'poly2':
            phi = template_poly2(mjd, t_max, s)
        else:
            phi = template_weibull(mjd, t_max, s, **kw)

        all_bands.append(bands_arr)
        all_mjd.append(mjd)
        all_flux.append(flux)
        all_ferr.append(ferr)
        all_phi.append(phi)
        all_Fpeak.append(np.full(len(mjd), F_pk))

    if not all_bands:
        return A_current

    bands_all  = np.concatenate(all_bands)
    flux_all   = np.concatenate(all_flux)
    ferr_all   = np.concatenate(all_ferr)
    phi_all    = np.concatenate(all_phi)
    Fpeak_all  = np.concatenate(all_Fpeak)

    p0 = [A_current[b] for b in color_bands]

    def chi2_colors(p):
        A_d = {ref_band: 1.0, **{b: p[i] for i, b in enumerate(color_bands)}}
        A_arr = np.array([A_d.get(b, 1.0) for b in bands_all])
        fp = A_arr * Fpeak_all * phi_all
        return float(np.sum(((flux_all - fp) / ferr_all)**2))

    bl = [0.01]*len(color_bands)
    bh = [20.0]*len(color_bands)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = minimize(chi2_colors, p0, method='L-BFGS-B',
                       bounds=list(zip(bl, bh)),
                       options={'maxiter': 500})
    return {b: res.x[i] for i, b in enumerate(color_bands)}


print("Global-fit helpers ready.")

In [ ]:
# ── Block-coordinate descent ──────────────────────────────────────────────────
MAX_ITER_GLOBAL = 10
CONV_TOL        = 1e-4    # relative change in A values for convergence

# Initialise shared colour factors (start from 1)
A_shared = {b: 1.0 for b in COLOR_BANDS}
per_event_global = {}

print("Starting block-coordinate descent...")
for iteration in range(MAX_ITER_GLOBAL):
    # Step A: update per-event parameters
    for snid in snids_global:
        r = fit_event_fixed_colors(
            ltcvs_global[snid], A_shared,
            model=FIT_MODEL, ref_band=REF_BAND, color_bands=COLOR_BANDS
        )
        per_event_global[snid] = r

    # Step B: update shared colour factors
    A_new = update_shared_colors(
        snids_global, ltcvs_global, per_event_global,
        A_shared, model=FIT_MODEL, ref_band=REF_BAND, color_bands=COLOR_BANDS
    )

    # Convergence check
    delta = max(abs(A_new[b] - A_shared[b]) / max(A_shared[b], 1e-6) for b in COLOR_BANDS)
    A_shared = A_new

    chi2_total = sum(
        v.get('chi2', 0) for v in per_event_global.values() if v.get('success')
    )
    print(f"  iter {iteration+1:2d}  |  ΔA_max={delta:.6f}  |  total χ²={chi2_total:.1f}")
    print(f"           A = " + "  ".join(f"{b}:{A_shared[b]:.4f}" for b in COLOR_BANDS))

    if delta < CONV_TOL:
        print(f"  → Converged after {iteration+1} iterations.")
        break

print("\n=== Final shared colour factors ===")
print(f"  A_{REF_BAND} = 1.0000  (reference)")
for b in COLOR_BANDS:
    print(f"  A_{b} = {A_shared[b]:.4f}")

In [ ]:
# ── Plot: global colour factors as a bar chart ─────────────────────────────────
all_bands_plot = BANDS
A_vals_plot = [1.0 if b == REF_BAND else A_shared.get(b, np.nan) for b in all_bands_plot]

fig, ax = plt.subplots(figsize=(7, 3.5), tight_layout=True)
bars = ax.bar(all_bands_plot, A_vals_plot,
              color=[BAND_COLORS.get(b, 'gray') for b in all_bands_plot],
              edgecolor='k', linewidth=0.8)
ax.axhline(1.0, color='k', ls='--', lw=1, label=f'{REF_BAND}-band reference')
for bar, val in zip(bars, A_vals_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('LSST band', fontsize=11)
ax.set_ylabel(r'$A_b$ (flux ratio relative to r-band)', fontsize=10)
ax.set_title(f'Global colour factors — {OBJ_CLASS}  ({FIT_MODEL} model,  N={n_global} events)', fontsize=11)
ax.legend(fontsize=9)
plt.show()

In [ ]:
# ── Plot: renormalised light curves with GLOBAL colour factors ─────────────────
# Show a random subsample of n_show events
n_show    = min(N_CURVES, n_global)
rng_show  = np.random.default_rng(seed=99)
show_idx  = rng_show.choice(n_global, size=n_show, replace=False)
show_snids = snids_global[show_idx]

nrows_p2 = math.ceil(n_show / NCOLS)
fig2, axes2 = plt.subplots(nrows_p2, NCOLS,
                            figsize=(5.5*NCOLS, 4.2*nrows_p2), tight_layout=True)
axes2_flat = np.array(axes2).flatten()

for idx, snid in enumerate(show_snids):
    ax  = axes2_flat[idx]
    ev  = per_event_global.get(snid, {})
    if not ev.get('F_peak'):
        ax.set_title(f"SNID {snid}\nFit failed", fontsize=9, color='red')
        continue

    t_max  = ev['t_max']; s = ev['s']; F_peak = ev['F_peak']
    z_val  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_val  = z_val[0] if len(z_val) else float('nan')
    chi2r  = ev.get('chi2', np.nan) / max(1, ltcvs_global[snid].shape[0] - 3)

    A_full_ev = {REF_BAND: 1.0, **{b: A_shared[b] for b in COLOR_BANDS}}

    for band in BANDS:
        bdf = ltcvs_global[snid]
        bdf = bdf[bdf['BAND'] == band]
        if len(bdf) == 0:
            continue
        A_b = A_full_ev.get(band, 1.0)
        norm_factor = max(A_b * F_peak, 1e-6)
        dt   = bdf['MJD'].values - t_max
        f_n  = bdf['FLUXCAL'].values / norm_factor
        fe_n = bdf['FLUXCALERR'].values / norm_factor
        ax.errorbar(dt, f_n, yerr=fe_n,
                    color=BAND_COLORS.get(band, 'gray'),
                    linestyle='None', marker='o', markersize=3,
                    capsize=2, label=band)

    kw_model = {}
    if FIT_MODEL == 'weibull':
        kw_model = {'beta': ev.get('beta', 2.0), 'gamma_asym': ev.get('gamma_asym', 1.5)}
    if FIT_MODEL == 'poly2':
        phi_m = template_poly2(t_max + t_model_dense, t_max, s)
    else:
        phi_m = template_weibull(t_max + t_model_dense, t_max, s, **kw_model)
    ax.plot(t_model_dense, phi_m, 'k-', lw=1.5, label='model', zorder=5)
    ax.axhline(0, color='k', lw=0.4, ls='--')

    ax.set_xlim(t_model_dense[0], t_model_dense[-1])
    ax.set_title(
        f"SNID {snid}  z={z_val:.3f}\n"
        f"t_max={t_max:.1f}  s={s:.1f} d  χ²/dof≈{chi2r:.2f}",
        fontsize=8
    )
    ax.set_xlabel(r"$t - t_{\rm max}$ [days]", fontsize=8)
    ax.set_ylabel(r"Flux / $(A_b F_{\rm peak})$", fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.legend(fontsize=6, ncol=3, loc='upper right')

for idx in range(n_show, len(axes2_flat)):
    axes2_flat[idx].set_visible(False)

fig2.suptitle(
    f"Part 2 · Global-fit renormalised light curves ({FIT_MODEL}) — {OBJ_CLASS}\n"
    f"Shared colour factors:  "
    + "  ".join(f"$A_{{\\rm {b}}}$={A_shared[b]:.3f}" for b in COLOR_BANDS),
    fontsize=10, y=1.01
)
plt.tight_layout()
plt.show()

## Part 2 — Distribution of per-event parameters after global fit

In [ ]:
# Collect per-event parameters
rows_p2 = []
for snid in snids_global:
    ev = per_event_global.get(snid, {})
    if not ev.get('F_peak'):
        continue
    z_val = truth[truth['SNID'] == snid]['ZCMB'].values
    z_val = z_val[0] if len(z_val) else float('nan')
    row = {'SNID': snid, 'z': z_val,
           'F_peak': ev['F_peak'], 't_max': ev['t_max'], 's': ev['s'],
           'chi2': ev.get('chi2', np.nan)}
    if FIT_MODEL == 'weibull':
        row['beta'] = ev.get('beta', np.nan)
        row['gamma_asym'] = ev.get('gamma_asym', np.nan)
    rows_p2.append(row)

df_p2 = pd.DataFrame(rows_p2)

params_to_plot = ['s', 'F_peak']
if FIT_MODEL == 'weibull':
    params_to_plot += ['beta', 'gamma_asym']

fig3, axes3 = plt.subplots(1, len(params_to_plot), figsize=(4.5*len(params_to_plot), 3.5),
                            tight_layout=True)
if len(params_to_plot) == 1:
    axes3 = [axes3]

labels = {'s': 'Stretch s [days]', 'F_peak': r'$F_{\rm peak}$ (r-band)',
          'beta': r'Weibull $\beta$', 'gamma_asym': r'Asymmetry $\gamma$'}
for ax, par in zip(axes3, params_to_plot):
    vals = df_p2[par].dropna()
    ax.hist(vals, bins=15, color='steelblue', edgecolor='white')
    ax.set_xlabel(labels.get(par, par), fontsize=10)
    ax.set_ylabel('N events', fontsize=10)
    ax.set_title(f'median = {np.median(vals):.3g}  σ = {np.std(vals):.3g}', fontsize=9)

fig3.suptitle(f'Per-event parameter distributions — global fit ({FIT_MODEL})  N={n_global}',
              fontsize=11)
plt.show()

## Summary

| | Part 1 | Part 2 |
|---|---|---|
| Colour factors $A_b$ | free per event | **shared** across all events |
| $F_{\rm peak}$, $t_{\rm max}$, $s$ | free per event | free per event |
| Weibull $\beta$, $\gamma$ | free per event | free per event |
| Optimiser | L-BFGS-B | block-coordinate descent |

The global colour factors $A_b$ represent an **average spectral energy distribution** of the
SNIa population in the chosen redshift range, under the assumption that the template shape
is the same in all bands up to a multiplicative constant.
